# API (Application Programming Interface)
다른 프로그램의 기능을 코드로 요청하는 것. 직접 모델을 학습하거나 서버에 띄우지 않고 google 서버에 요청
> google-genai SDK -> gemini-2.5-flash 를 사용

* API 키 발급 — Google AI Studio에서 무료로 발급.
* 첫 호출 — `client.models.generate_content(...)` 한 줄.
* 응답 다루기 — `response.text(답변)`, `response.usage_metadata(토큰=비용)`.


### 0. 설치

In [ ]:
!pip install -q google-genai   # 코랩엔 기본 미설치 → 구글 신 SDK 설치(-q: 설치 로그를 줄임)

### 1. API key 설정
* [Google AI Studio](https://aistudio.google.com/app/api-keys) 접속 → Create API key → 키 복사.
* 코랩 왼쪽 🔑 보안 비밀(Secrets) → 새 비밀 추가   
→ 이름 GEMINI_API_KEY, 값에 키 붙여넣기 → 노트북 액세스 ON.

In [1]:
import os
from google.colab import userdata   # 코랩 'Secrets'(🔑)에 저장한 비밀값을 읽는 도구
# Secrets 에 저장한 API 키를 읽어 환경변수(os.environ)로 올린다.
# 키 이름을 GEMINI_API_KEY → GOOGLE_API_KEY 순서로 시도(둘 중 무엇으로 저장했든 동작하게)
for _n in ('GEMINI_API_KEY', 'GOOGLE_API_KEY'):
    try:
        os.environ['GEMINI_API_KEY'] = userdata.get(_n)  # 읽기 성공 시 환경변수에 저장
        break                                            # 하나라도 성공하면 반복 종료
    except Exception:
        pass                                             # 그 이름이 없으면 다음 후보로

### 2. 클라이언트 생성
* `client.models.generate_content` : Gemini에 '한 번 묻고 한 번 답받는' 가장 기본 호출

In [ ]:
from google import genai   # google-genai SDK 진입점(구 google-generativeai 아님)
# genai.Client : Gemini 서버와 통신하는 클라이언트 객체 — 앞으로 모든 호출의 시작점.
# api_key 를 직접 넘긴다(SDK가 환경변수를 자동으로 읽기도 하지만,
#  만료된 다른 키와 섞이는 혼선을 막으려고 우리가 명시적으로 지정)
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

### 3. 호출
* `response.text` — 생성된 텍스트

In [ ]:
# generate_content : 프롬프트를 보내고 답을 받는 한 번의 호출
response = client.models.generate_content(
    model='gemini-2.5-flash',   # 빠르고 저렴한 기본 모델
    # contents : 모델에 보낼 입력 텍스트. '역할(고객센터 AI) + 상황 + 요청(한 문장 정중히)'
    contents='너는 쇼핑몰 고객센터 AI야. "배송이 너무 늦어요"라는 문의에 한 문장으로 정중히 답해줘.',
)
print(response.text)   # response.text : 생성된 답변 텍스트(가장 자주 쓰는 값)

배송 지연으로 불편을 드린 점 진심으로 사과드리며, 고객님의 주문 내역을 확인하여 신속히 안내해 드리겠습니다.


### 4. 토큰 사용량 확인
* `response.usage_metadata` — 토큰 사용량(입력/출력/합계) -> 비용과 연결됨.

In [ ]:
u = response.usage_metadata   # usage_metadata : 이번 호출이 쓴 토큰량 정보 묶음
print('입력 토큰 :', u.prompt_token_count)       # 우리가 보낸 프롬프트의 토큰 수
print('출력 토큰 :', u.candidates_token_count)   # 모델이 생성한 답변의 토큰 수
print('합계      :', u.total_token_count)        # 입력+출력 = 과금 기준(비용 계산의 근거)

입력 토큰 : 34
출력 토큰 : 34
합계      : 1188


### 5. 그외
* `response.candidates`, `response.parts` — 후보·세부 파트(구조화 출력·멀티모달에서 활용)


# 실습 문제

## 문제 1 — 질문 함수 ask
프롬프트 문자열을 받아 `gemini-2.5-flash-lite` 로 호출하고 답변 텍스트만 돌려주는 `ask(prompt)` 를 작성하세요. 서로 다른 CS 문의 2개로 호출해 보세요.

In [7]:
from google import genai
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

model='gemini-2.5-flash-lite'

def ask(prompt):
    response = client.models.generate_content(
        model=model,
        contents=prompt,
    )
    return response.text

In [8]:
ask('반품은 며칠 이내에 가능한가요? 한문장으로')

'상품 수령일로부터 7일 이내에 반품이 가능합니다.'

In [9]:
ask('교환 배송비는 얼마인가요? 한문장으로')

'교환 배송비는 상품 종류와 교환 사유에 따라 다르며, 일반적으로 왕복 배송비가 발생합니다.'

-> 호출을 함수로 감싸면 매번 `model=`·`contents=` 를 쓰지 않아도 됩니다. 이 패턴이 공통 유틸에 사용됩니다.

## 문제 2 — 토큰까지 함께 반환
답변 텍스트와 총 토큰 수를 함께 돌려주는 `ask_with_usage(prompt)` → `(text, total_tokens)` 를 작성하세요.



In [10]:
from google import genai
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

model='gemini-2.5-flash-lite'

def ask_with_usage(prompt):
    response = client.models.generate_content(
        model=model,
        contents=prompt,
    )
    return response.text, response.usage_metadata.total_token_count

In [12]:
text, token_cnt = ask_with_usage('배송비는 얼마인가요? 한문장으로')
print(token_cnt, 'token ->', text)

28 token -> 배송비는 상품 가격과 배송 방법에 따라 달라집니다.


## 문제 3 (심화) — 비용 추정
입력/출력 토큰 수와 1,000토큰당 단가를 받아 예상 비용을 계산하는 `estimate_cost(in_tok, out_tok, in_price, out_price)` 를 작성하세요.

In [13]:
def estimate_cost(in_tok, out_tok, in_price, out_price):
    return (in_tok / 1000) * in_price + (out_tok / 1000) * out_price

In [14]:
# 가상 단가(1K토큰당 입력 0.075 / 출력 0.30 달러) 예시
print('예상 비용($):', estimate_cost(1200, 300, 0.075, 0.30))

예상 비용($): 0.18


-> (토큰 수) x (단가)로 호출당 비용을 추정하면, 대량 처리 시 총비용을 가늠할 수 있음.